# ETF Universe Cleaning & Data Pipeline
## QF 623 Portfolio Management — Group Project

This notebook takes a pre-screened ETF master universe and produces clean, analysis-ready return panels for **Bond** and **Sector** ETFs.

### What this notebook does (in order)
1. Load the cleaned ETF universe and raw CRSP market data
2. Apply final quality guards (leveraged ETFs, ETNs, etc.)
3. Resolve a CRSP security identifier (`permno`) for each ETF via a 3-stage match
4. Download daily returns from CRSP for every ETF in the universe
5. Compute return and volatility statistics, net of expense ratios
6. Deduplicate near-identical ETFs (e.g. SPY / IVV / VOO all track the S&P 500)
7. Classify ETFs into types (Sector, Treasury, Corporate Bond, etc.)
8. Build separate return panels for Bond and Sector subsets
9. Save all outputs to `data/processed/`

In [ ]:
import pandas as pd   # tabular data — DataFrames, series operations
import numpy as np    # numerical operations — sqrt, log, prod, etc.
import os             # file system operations — creating directories, checking paths

# Root folder for all project data — raw inputs and processed outputs live here
DATA_DIR = r"D:\SMU\Course Work\QF 623 - Portfolio Management\Group Project\etf_long_short_project\data"

# Where we write cleaned outputs (parquet files) after processing
PROC_DIR = os.path.join(DATA_DIR, "processed")

# exist_ok=True means: create the folder if it doesn't exist, do nothing if it does
os.makedirs(PROC_DIR, exist_ok=True)

## 1. Load Data

We load three datasets:

- **ETF universe** (`clean_etf_master_universe.csv`) — 1,630 ETFs that passed the initial screening. Contains metadata like AUM, expense ratio, benchmark, and data-quality flags.
- **CRSP daily** — daily returns, prices, and volume from CRSP's stock database. This is our source of truth for historical performance.
- **CRSP names** — maps each security to a CRSP numeric identifier called `permno`. We use this to link the universe to the returns data.

In [ ]:
# Pre-screened ETF universe — the starting point for all subsequent steps
universe = pd.read_csv(r"D:\SMU\Course Work\QF 623 - Portfolio Management\Group Project\clean_etf_master_universe.csv")

# Daily returns file from CRSP — the original pull (will be replaced in Step 4
# with a broader pull that covers all permnos in this new universe)
daily = pd.read_parquet(f"{DATA_DIR}/raw/crsp_etf_daily.parquet")

# CRSP name history — maps securities to their permno identifier
names = pd.read_parquet(f"{DATA_DIR}/raw/crsp_etf_names.parquet")

# Ensure correct types — Parquet preserves most types, but being explicit avoids
# silent bugs later (e.g. date comparisons failing because dates are strings)
daily['date'] = pd.to_datetime(daily['date'])
daily['ret']  = pd.to_numeric(daily['ret'], errors='coerce')  # 'C' (CRSP missing code) → NaN

print(f"Universe : {len(universe):,} ETFs")
print(f"Daily    : {len(daily):,} rows  |  {daily['date'].min().date()} – {daily['date'].max().date()}")
print(f"Names    : {len(names):,} rows")

## 2. Apply Quality Guards

The universe file has already been through a first-pass screen, so most flag columns are uniformly clean. We apply two final hard guards:

- **`is_leveraged == False`** — removes ETFs that use leverage (e.g. 2× or 3× products). These amplify index returns using derivatives and behave very differently from regular ETFs.
- **`is_etn == 0`** — removes Exchange-Traded Notes. ETNs are debt instruments issued by banks, not investment funds. They carry **counterparty risk** (if the bank fails, you lose your money) that ETFs don't have.

> **Why not filter `is_active`?**
> `is_active == 0` means the ETF was eventually *delisted*, not that it was invalid during our backtest window. Filtering these out would introduce **survivorship bias** — we'd only be looking at ETFs that survived, which artificially inflates strategy performance. We keep all ETFs and let their price history naturally end at the delisting date.

In [ ]:
# The universe file is already pre-screened, so most flag columns are uniformly clean.
# We apply two final hard guards as a safety check.
#
# is_leveraged == False  →  drops 2×/3× leveraged ETFs (they use swaps/futures to amplify
#                            index returns and behave like derivatives, not normal ETFs)
#
# is_etn == 0            →  drops Exchange-Traded Notes (bank IOUs with counterparty risk)
#
# NOTE: We do NOT filter on is_active.
#   is_active == 0 means the ETF was eventually delisted — not that it never existed.
#   Excluding delisted ETFs would cause survivorship bias: our backtest would only
#   look at ETFs that happened to survive, making past performance look better than
#   it actually was.

df = universe[
    (universe['is_leveraged'] == False) &
    (universe['is_etn'] == 0)
].copy()

print(f"After guards: {len(df):,} ETFs  (dropped {len(universe) - len(df):,})")
print(f"  Currently active (still listed) : {(df['is_active'] == 1).sum()}")
print(f"  Inactive (delisted at some point): {(df['is_active'] == 0).sum()}")

## 3. Resolve CRSP Security Identifiers (permno)

CRSP identifies every security by a number called **`permno`**. Our universe file doesn't have these — they're all null — so we need to derive them by matching against CRSP's name history table.

### Why not just match on ticker?
Tickers get **recycled**. A ticker that belonged to a defunct company in 2005 might belong to a completely different ETF today. A straight ticker match can silently assign the wrong `permno`.

### Our solution: 3-stage cascade

We run three match attempts from most reliable to least:

| Stage | Match key | Filter | Why |
|-------|-----------|--------|-----|
| 1 | CUSIP (8-char) | none | CUSIPs are security-specific and never recycled — the gold standard identifier |
| 2 | Ticker | `shrcd == 73` (ETF share class only) | Avoids recycled tickers by requiring the CRSP record to also be coded as an ETF |
| 3 | Ticker | Any share code | Last resort — some older ETFs were coded under non-73 share classes early in CRSP history |

We first re-query WRDS with a broader pull (CUSIP **OR** ticker, no share-code restriction) to get the richest possible names table.

In [ ]:
# ── ONE-TIME WRDS PULL ─────────────────────────────────────────────────────────
# We re-query WRDS with a broader filter than the original names pull:
#   - No share-code restriction (catches ETFs coded under non-73 share classes)
#   - Match on CUSIP OR ticker (two chances to find each security)
# After running once, the result is saved to parquet — skip this cell on re-runs.
import wrds

def to_cusip8(series):
    """
    Convert a CUSIP column to clean 8-character strings.

    The problem: pandas sometimes reads CUSIPs as floats (e.g. '594918104' becomes
    594918104.0), which adds a '.0' suffix and can lose leading zeros.

    The fix:
      1. Cast to string           → '594918104.0'
      2. Strip the '.0' suffix    → '594918104'
      3. Zero-pad to 9 characters → '594918104' (no change here; needed if leading zeros lost)
      4. Take first 8 characters  → '59491810'  (CRSP uses 8-char CUSIPs, not 9)
    """
    return (
        series.dropna()
        .astype(str)
        .str.replace(r'\.0$', '', regex=True)   # remove float artefact
        .str.strip()
        .str.zfill(9)                            # restore any leading zeros lost to float conversion
        .str[:8]                                 # CRSP stores 8-char CUSIPs (no check digit)
        .unique()
        .tolist()
    )

def sql_in(lst):
    """
    Build a SQL IN(...) string from a Python list, safely handling edge cases.

    Why not just use str(tuple(lst))?
      - Single-item tuple → ('X',)  ← trailing comma breaks PostgreSQL
      - Empty list        → ()      ← empty IN clause also breaks PostgreSQL
    This helper avoids both problems by building the string manually.
    """
    if not lst:
        return None
    escaped = ", ".join(f"'{v}'" for v in lst)
    return f"({escaped})"

cusips  = to_cusip8(universe['cusip'])
tickers = universe['ticker'].dropna().unique().tolist()

print(f"CUSIPs to query : {len(cusips)}")
print(f"Tickers to query: {len(tickers)}")

# Build WHERE clause dynamically — only include a condition if the list is non-empty
parts = []
if cusips:
    parts.append(f"cusip IN {sql_in(cusips)}")
if tickers:
    parts.append(f"ticker IN {sql_in(tickers)}")

if not parts:
    raise ValueError("Both cusip and ticker lists are empty — nothing to query.")

where_clause = " OR ".join(parts)

conn       = wrds.Connection()
names_full = conn.raw_sql(f"""
    SELECT permno, cusip, ticker, namedt, nameendt, shrcd, exchcd, comnam
    FROM crsp.dsenames
    WHERE {where_clause}
""")
conn.close()

names_full.to_parquet(f"{DATA_DIR}/raw/crsp_etf_names_full.parquet", index=False)
print(f"\nPulled {len(names_full):,} name records  |  {names_full['permno'].nunique():,} unique permnos")
print(f"shrcd distribution:\n{names_full['shrcd'].value_counts().head(10).to_string()}")

In [ ]:
# Load the broader names table pulled from WRDS above
names_full = pd.read_parquet(f"{DATA_DIR}/raw/crsp_etf_names_full.parquet")
names_full['namedt'] = pd.to_datetime(names_full['namedt'])

# ── CUSIP normalisation for column assignment ──────────────────────────────────
# to_cusip8() (defined in the WRDS pull cell) returns a deduplicated LIST for SQL.
# Here we need a Series that stays aligned with the DataFrame index, so we use
# a separate inline operation that does NOT call .unique().tolist() at the end.
def normalise_cusip_col(series):
    """
    Convert a CUSIP column to 8-char strings, keeping the original index intact.
    Unlike to_cusip8(), this returns a Series (not a list) so it can be assigned
    directly as a new DataFrame column.
    """
    return (
        series.astype(str)                          # cast to string (handles float CUSIPs)
        .str.replace(r'\.0$', '', regex=True)       # strip float artefact '594918104.0' → '594918104'
        .str.strip()
        .str.zfill(9)                               # restore leading zeros to get 9 chars
        .str[:8]                                    # keep first 8 chars (CRSP format)
        .where(series.notna(), other=np.nan)        # restore NaN where original was NaN
    )

df['cusip_8']        = normalise_cusip_col(df['cusip'])
names_full['cusip8'] = normalise_cusip_col(names_full['cusip'].fillna(np.nan))

# ── Build lookup dictionaries: identifier → permno ────────────────────────────
# .last() picks the most recent record when one security has multiple name entries
# (e.g. a fund that changed its name but kept the same permno — common in ETFs)

# Stage 1: CUSIP → permno
# Most reliable: CUSIPs are security-specific and never reassigned to another fund
cusip_to_permno = (
    names_full[names_full['cusip8'].notna() & (names_full['cusip8'].str.len() == 8)]
    .sort_values('namedt')
    .groupby('cusip8')['permno'].last()
)

# Stage 2: ticker → permno, ETF records only (shrcd = 73)
# shrcd = 73 is CRSP's share-code for ETFs — this prevents matching a recycled
# ticker that previously belonged to a regular stock
ticker_to_permno_73 = (
    names_full[names_full['shrcd'] == 73]
    .sort_values('namedt')
    .groupby('ticker')['permno'].last()
)

# Stage 3: ticker → permno, any share class
# Broadest fallback — some ETFs launched before CRSP adopted shrcd=73 and were
# coded under other share classes (e.g. 11 or 12)
ticker_to_permno_any = (
    names_full
    .sort_values('namedt')
    .groupby('ticker')['permno'].last()
)

# ── Apply the cascade ─────────────────────────────────────────────────────────
# combine_first(b): use values from self where not NaN, otherwise use values from b
# Chaining two combine_first calls implements the three-stage fallback logic
s1 = df['cusip_8'].map(cusip_to_permno)        # Stage 1 results (NaN where no match)
s2 = df['ticker'].map(ticker_to_permno_73)     # Stage 2 results (NaN where no match)
s3 = df['ticker'].map(ticker_to_permno_any)    # Stage 3 results (NaN where no match)

df['permno'] = s1.combine_first(s2).combine_first(s3)

# ── Coverage report ───────────────────────────────────────────────────────────
n1     = s1.notna().sum()
n2     = (s1.isna() & s2.notna()).sum()              # matched only in Stage 2
n3     = (s1.isna() & s2.isna() & s3.notna()).sum()  # matched only in Stage 3
n_miss = (s1.isna() & s2.isna() & s3.isna()).sum()   # failed all three stages

print(f"Stage 1 — CUSIP match              : {n1:>4}")
print(f"Stage 2 — ticker match (shrcd=73)  : {n2:>4}  (additionally)")
print(f"Stage 3 — ticker match (any shrcd) : {n3:>4}  (additionally)")
print(f"──────────────────────────────────────────")
print(f"Unmatched                          : {n_miss:>4}  / {len(df)}")

if n_miss > 0:
    print("\nStill unmatched — review these:")
    print(df[df['permno'].isna()][['ticker','fund_name','cusip','asset_class']].to_string(index=False))

## 4. Download Daily Returns from CRSP

The original `crsp_etf_daily.parquet` was built for an earlier, smaller universe and is missing most of our 1,630 ETFs' permnos — which is why a naive history filter dropped 1,597 ETFs.

We fix this by re-pulling daily data from WRDS for **every permno in the new universe**.

> **This cell only needs to run once.** It saves `crsp_etf_daily_new.parquet` to disk. Skip it on subsequent runs.

**Fields pulled from `crsp.dsf`:**

| Field | Meaning |
|-------|---------|
| `ret` | Daily return — already adjusted by CRSP for stock splits and dividends |
| `prc` | Closing price |
| `vol` | Trading volume (number of shares) |
| `shrout` | Shares outstanding — used to compute market cap |

In [ ]:
# ── ONE-TIME WRDS PULL — daily returns for the full new universe ───────────────
# The original crsp_etf_daily.parquet only covered the permnos from an earlier,
# smaller universe. Most of the 1,630 permnos in our new universe were absent,
# which caused the history filter to drop ~1,600 ETFs.
#
# We fix this by pulling fresh daily data for every permno we resolved above.
# After running once, the result is saved to parquet — skip this cell on re-runs.
import wrds

# Collect every permno we successfully matched, cast to int (CRSP uses integers)
new_permnos = df['permno'].dropna().astype(int).unique().tolist()
print(f"Pulling daily data for {len(new_permnos):,} permnos — this may take a few minutes ...")

conn = wrds.Connection()
daily_new = conn.raw_sql(f"""
    SELECT permno, date, ret, prc, vol, shrout
    FROM crsp.dsf
    WHERE permno IN {sql_in([str(p) for p in new_permnos])}
      AND date BETWEEN '2011-01-01' AND CURRENT_DATE
""")
conn.close()

daily_new.to_parquet(f"{DATA_DIR}/raw/crsp_etf_daily_new.parquet", index=False)

print(f"Saved {len(daily_new):,} rows  |  "
      f"{daily_new['permno'].nunique():,} unique permnos  |  "
      f"{daily_new['date'].min()} – {daily_new['date'].max()}")

## 5. Filter to Backtest Window & Compute Net-of-Fee Returns

We slice the daily data to our backtest window (**January 2011 → present**) and adjust each day's return for the ETF's **expense ratio drag**.

### Why adjust for the expense ratio?
Every ETF silently deducts its annual management fee from NAV on a daily basis. Without adjusting for this, comparing a cheap ETF (e.g. VOO at 0.03%/year) against an expensive one (e.g. 0.50%/year) is like comparing two runners where one is carrying a backpack — the comparison isn't fair.

**Formula applied to every daily return:**

$$r^{net}_t = r_t - \frac{\text{net\_expenses}}{252}$$

We divide the annual fee by 252 (trading days per year) to get the daily drag, then subtract it from the raw return.

> `net_expenses` is the all-in annual fee **after** any temporary fee waivers. This is the number investors actually pay.

In [ ]:
BACKTEST_START = '2011-01-01'

# Load the freshly-pulled daily panel (covers all permnos in the new universe)
daily = pd.read_parquet(f"{DATA_DIR}/raw/crsp_etf_daily_new.parquet")
daily['date'] = pd.to_datetime(daily['date'])
daily['ret']  = pd.to_numeric(daily['ret'], errors='coerce')  # 'C' → NaN

# Keep only permnos in our filtered universe and dates within the backtest window
df['permno']  = pd.to_numeric(df['permno'], errors='coerce')
valid_permnos = df['permno'].dropna().astype(int).tolist()

daily_f = daily[
    (daily['date'] >= BACKTEST_START) &
    (daily['permno'].isin(valid_permnos))
].copy()

# ── Expense ratio adjustment ───────────────────────────────────────────────────
# Each ETF deducts its annual fee from NAV every single trading day.
# We replicate this by subtracting (annual_fee / 252) from each daily return.
#
# Example: A fund with net_expenses = 0.0009 (0.09%/year, like SPY)
#          deducts 0.0009 / 252 ≈ 0.0000036 (0.00036%) from each day's return.
#
# This makes returns comparable across cheap and expensive ETFs.

# Map permno → net_expenses, then merge into the daily data
expense_map = (
    df[['permno', 'net_expenses']]
    .dropna(subset=['permno'])
    .assign(permno=lambda x: x['permno'].astype(int))
)
daily_f = daily_f.merge(expense_map, on='permno', how='left')
daily_f['net_expenses'] = daily_f['net_expenses'].fillna(0)  # treat unknown fees as 0
daily_f['ret_net'] = daily_f['ret'] - daily_f['net_expenses'] / 252

print(f"Daily obs after filter: {len(daily_f):,}  |  ETFs in panel: {daily_f['permno'].nunique()}")

## 6. Compute Per-ETF Summary Statistics

For each ETF we compute three statistics over the **full backtest period** (2011 → present):

| Statistic | Formula | What it tells you |
|-----------|---------|-------------------|
| `ann_return` | $(1+r_1)(1+r_2)\cdots(1+r_n)^{252/n} - 1$ | Compound annual growth rate (CAGR), net of fees |
| `daily_vol` | $\text{std}(r_t)$ | How much the ETF's daily return fluctuates |
| `ann_vol` | $\text{daily\_vol} \times \sqrt{252}$ | Annualised risk — the denominator in the Sharpe ratio |

> ETFs with fewer than **252 trading days** of data are flagged via `n_obs` and removed in the next step. 252 ≈ 1 trading year, the minimum needed to estimate statistics reliably.

In [ ]:
def etf_stats(grp):
    """
    Compute full-period return and risk statistics for a single ETF.

    Parameters
    ----------
    grp : DataFrame
        All daily rows for one permno. Must have a 'ret_net' column.

    Returns
    -------
    pandas Series with four fields:
        ann_return  — compound annual growth rate (CAGR) net of fees
        ann_vol     — annualised volatility (standard deviation × √252)
        daily_vol   — raw daily standard deviation (before annualising)
        n_obs       — number of valid return observations (NaN rows excluded)
    """
    r = grp['ret_net'].dropna()   # drop NaN return days (e.g. trading halts)
    n = len(r)

    # Require at least 1 year of data before computing statistics.
    # Below this threshold, estimates are too noisy to be meaningful.
    if n < 252:
        return pd.Series(dict(ann_return=np.nan, ann_vol=np.nan, daily_vol=np.nan, n_obs=n))

    # CAGR: multiply all (1 + daily_return) factors, then raise to the power (252/n).
    # This converts a multi-year cumulative return into an equivalent annual rate.
    # Example: $1 growing to $2 over 5 years → (2)^(1/5) - 1 ≈ 14.9%/year
    ann_ret = (1 + r).prod() ** (252 / n) - 1

    d_vol = r.std()   # daily standard deviation

    return pd.Series(dict(
        ann_return = ann_ret,
        ann_vol    = d_vol * np.sqrt(252),   # annualise: scale up by √252 trading days
        daily_vol  = d_vol,
        n_obs      = n
    ))

# Apply to each ETF — groupby('permno') splits the data, etf_stats() runs on each piece
stats = daily_f.groupby('permno').apply(etf_stats).reset_index()
stats['permno'] = stats['permno'].astype(int)
print(stats[['ann_return', 'ann_vol', 'daily_vol']].describe().round(4))

## 7. Enforce Minimum History (≥ 1 Year)

We keep only ETFs with at least **252 daily return observations** within the backtest window. ETFs below this threshold either:

- Launched after the backtest start date, or
- Were delisted very early and have almost no tradeable history

Dropping them prevents the strategy from relying on ETFs with statistically unreliable estimates — you can't draw meaningful conclusions from a few months of returns.

In [ ]:
# Int64 (capital I) is pandas' nullable integer type — it can hold NaN values.
# Regular int64 (lowercase) cannot. We use Int64 here because some permnos may still
# be null after the cascade match, and a type mismatch would silently break the merge.
df['permno']    = df['permno'].astype('Int64')
stats['permno'] = stats['permno'].astype('Int64')

# Join the computed statistics back onto the universe metadata table.
# left join: every ETF in df is retained, NaN stats for those without daily data.
df = df.merge(stats, on='permno', how='left')

# Drop ETFs with fewer than 252 return observations (< 1 trading year).
# These launched too recently or closed too early to have reliable statistics.
before = len(df)
df     = df[df['n_obs'] >= 252].copy()
print(f"Dropped {before - len(df)} ETFs with < 1 year of data  |  Remaining: {len(df)}")

## 8. Deduplicate Near-Identical ETFs

Many issuers offer ETFs tracking the **exact same index** — SPY, IVV, and VOO all replicate the S&P 500. Keeping all three would:
- Artificially over-weight the S&P 500 in the universe
- Treat three identical exposures as independent signals when they're not

We remove duplicates in two stages:

### Stage A — Same benchmark → keep most liquid
Group by `primary_benchmark`. Within each group, keep the ETF with the highest **`recent_median_dollar_volume`** (recent daily dollar trading volume).

> **Why dollar volume, not AUM?**
> AUM measures how much money is *invested* in the fund — a stock metric.
> Dollar volume measures how much can be *traded* in a day — a flow metric.
> For a strategy that needs to enter and exit positions, liquidity is what matters.
> SPY trades ~$33B/day vs IVV's ~$4B/day, so SPY wins on execution quality even though IVV has higher AUM.

### Stage B — Correlation-based catch-all
Some ETFs have no `primary_benchmark` entry. For these, we compute pairwise return correlations within the same `category`. If two ETFs have correlation > 0.97 (returns are virtually identical), they're treated as duplicates and the less liquid one is dropped.

In [ ]:
# Split the universe based on whether a primary_benchmark is recorded.
# ETFs with a known benchmark → Stage A (faster, more reliable)
# ETFs without a benchmark   → Stage B (correlation-based, slower)
df_bmark    = df[df['primary_benchmark'].notna()].copy()
df_no_bmark = df[df['primary_benchmark'].isna()].copy()

# ── Stage A: Same benchmark → keep the most liquid one ────────────────────────
# Logic: sort by recent_median_dollar_volume (descending), then .first() keeps
# the top-ranked ETF within each primary_benchmark group.
#
# Why recent_median_dollar_volume instead of AUM?
#   AUM = stock of assets under management (how popular the fund is overall)
#   Dollar volume = flow of daily trading (how easily you can buy/sell TODAY)
#   For a trading strategy, execution quality is what matters — and SPY trades
#   ~$33B/day vs IVV's ~$4B/day, so SPY is the right S&P 500 representative
#   even though IVV has a slightly higher AUM.
df_deduped_a = (
    df_bmark
    .sort_values('recent_median_dollar_volume', ascending=False, na_position='last')
    .groupby('primary_benchmark', as_index=False)
    .first()
)

print(f"Benchmark dedup: {len(df_bmark)} → {len(df_deduped_a)} ETFs  (removed {len(df_bmark) - len(df_deduped_a)})")
print(f"No-benchmark ETFs going to Stage B: {len(df_no_bmark)}")

# Spot-check: confirm the most liquid representative was selected for key benchmarks
spot = df_deduped_a[df_deduped_a['ticker'].isin(['SPY','IVV','VOO','QQQ','TLT','GLD'])]
print(f"\nSpot-check — kept representatives for key benchmarks:")
print(spot[['ticker','fund_name','recent_median_dollar_volume','aum','primary_benchmark']].to_string(index=False))

In [ ]:
to_drop = set()   # will accumulate permnos flagged for removal

if len(df_no_bmark) > 0:
    # Build a wide return matrix: rows = dates, columns = permnos
    # This is needed to compute pairwise correlations between ETFs
    nb_permnos = df_no_bmark['permno'].dropna().astype(int).tolist()
    ret_pivot  = (
        daily_f[daily_f['permno'].isin(nb_permnos)]
        .pivot_table(index='date', columns='permno', values='ret_net')
    )

    # Pairwise return correlations — if two ETFs have correlation > 0.97,
    # their daily returns are virtually identical, meaning they offer the same exposure
    corr = ret_pivot.corr()

    # Look up category and liquidity for each permno (used in the comparison below)
    cats = df_no_bmark.set_index('permno')['category'].to_dict()
    liq  = df_no_bmark.set_index('permno')['recent_median_dollar_volume'].fillna(0).to_dict()

    perms = [p for p in nb_permnos if p in corr.index]

    # O(n²) pairwise comparison — checks every combination of two ETFs.
    # Fine for a few hundred ETFs; would need optimisation for thousands.
    for i, p1 in enumerate(perms):
        if p1 in to_drop:
            continue   # already marked for removal — skip
        for p2 in perms[i+1:]:
            if p2 in to_drop:
                continue
            if cats.get(p1) != cats.get(p2):
                continue   # only compare within the same category (apples to apples)
            if corr.loc[p1, p2] > 0.97:
                # These two ETFs are near-identical in returns.
                # Drop the less liquid one (lower daily $ volume).
                to_drop.add(p1 if liq.get(p1, 0) < liq.get(p2, 0) else p2)

    df_no_bmark_d = df_no_bmark[~df_no_bmark['permno'].isin(to_drop)]
    print(f"Correlation dedup removed {len(to_drop)} ETFs  |  Remaining: {len(df_no_bmark_d)}")
else:
    df_no_bmark_d = df_no_bmark
    print("No no-benchmark ETFs to process in Stage B.")

# Combine the two deduplication results into one clean universe
df_clean = pd.concat([df_deduped_a, df_no_bmark_d], ignore_index=True)
print(f"\nFinal deduplicated universe: {len(df_clean)} ETFs")

## 9. Classify ETFs into Types

We assign each ETF to a type bucket using keyword matching on its `category` and `focus` columns. This lets us build separate sub-universes for different legs of the strategy.

**Equity types:**
| Type | Description |
|------|-------------|
| `Sector Equity` | Targets a specific GICS sector (Tech, Healthcare, Energy, etc.) |
| `Factor / Smart Beta` | Targets a style factor (momentum, value, quality, low-vol, dividend, etc.) |
| `International Equity` | Non-US developed and emerging market equity |
| `US Equity Broad` | Plain-vanilla US market-cap ETFs (SPY, IVV, etc.) |

**Bond types:**
| Type | Description | In strategy? |
|------|-------------|-------------|
| `Bond - Treasury` | US nominal government bonds (short / intermediate / long) | ✅ Yes |
| `Bond - Corporate` | Investment-grade corporate bonds | ✅ Yes |
| `Bond - High Yield` | Sub-investment-grade ("junk") bonds | ✅ Yes |
| `Bond - Aggregate` | Blended bond market (e.g. AGG) | ✅ Yes |
| `Bond - TIPS` | Inflation-linked Treasuries | ❌ Excluded (not nominal) |
| `Bond - Municipal` | Tax-exempt municipal bonds | ❌ Excluded |

> **Check the output carefully.** If `Other` is large (> ~5% of the universe), the keyword lists in the `classify()` function may not match this dataset's exact `category`/`focus` strings and will need refinement.

In [ ]:
# ── Keyword lists used by classify() below ────────────────────────────────────
# These are matched against a concatenation of the 'category' and 'focus' columns.
# Add or remove keywords here if the value_counts() output shows unexpected 'Other'.

SECTORS = [
    'technology', 'healthcare', 'financials', 'energy', 'materials',
    'industrials', 'consumer discretionary', 'consumer staples',
    'utilities', 'real estate', 'communication'
]   # GICS (Global Industry Classification Standard) sector names

FACTORS = [
    'momentum', 'quality', 'value', 'growth', 'dividend',
    'low volatility', 'minimum variance', 'multi-factor', 'smart beta'
]   # style tilts used by factor / smart-beta ETFs

INTL = [
    'international', 'global ex', 'emerging', 'developed ex',
    'europe', 'asia', 'japan', 'china', 'india', 'latam'
]   # keywords indicating a non-US geographic focus


def classify(row):
    """
    Assign an ETF to one of the type buckets defined in the markdown above.

    How it works:
      1. Pull asset_class, category, and focus — all converted to lowercase
      2. Concatenate category + focus into a single search string ('blob')
      3. Check fixed-income conditions first, then equity, then alternatives
      4. Return the first matching label; fall through to 'Other' if nothing matches

    Important: order of checks matters.
      Within bonds, 'inflation' is checked BEFORE 'treasury' because many TIPS
      descriptions also contain the word 'treasury'. First match wins.
    """
    a    = str(row.get('asset_class', '')).lower()
    c    = str(row.get('category',   '')).lower()
    f    = str(row.get('focus',      '')).lower()
    blob = c + ' ' + f   # search both columns at once to avoid missing edge cases

    # ── Fixed Income ──────────────────────────────────────────────────────────
    if 'fixed income' in a or 'bond' in a:
        if any(x in blob for x in ['inflation', 'tips', 'real return']):
            return 'Bond - TIPS'          # checked first — TIPS descriptions often include 'treasury'
        if any(x in blob for x in ['treasury', 'government', 'govt']):
            return 'Bond - Treasury'
        if any(x in blob for x in ['corporate', 'investment grade', ' ig ', 'credit']):
            return 'Bond - Corporate'
        if any(x in blob for x in ['high yield', 'junk', ' hy ']):
            return 'Bond - High Yield'
        if any(x in blob for x in ['muni', 'municipal']):
            return 'Bond - Municipal'
        if any(x in blob for x in ['aggregate', 'total bond']):
            return 'Bond - Aggregate'
        return 'Bond - Other'             # catch-all for fixed income not matched above

    # ── Equity ────────────────────────────────────────────────────────────────
    if 'equity' in a or 'stock' in a:
        if any(s in blob for s in SECTORS):
            return 'Sector Equity'
        if any(x in blob for x in FACTORS):
            return 'Factor / Smart Beta'
        if any(x in blob for x in INTL):
            return 'International Equity'
        return 'US Equity Broad'          # plain-vanilla US market-cap ETFs

    # ── Alternatives ──────────────────────────────────────────────────────────
    if any(x in a + blob for x in ['commodity', 'gold', 'silver', 'oil', 'metal']):
        return 'Commodity'
    if any(x in a + blob for x in ['currency', 'forex']):
        return 'Currency'
    if 'real estate' in a + blob or 'reit' in blob:
        return 'Real Estate'

    return 'Other'   # review this bucket — keywords above may need updating


df_clean['etf_type'] = df_clean.apply(classify, axis=1)

# If 'Other' is large (> ~5% of the universe), inspect those rows and
# add the missing keywords to the SECTORS / FACTORS / INTL lists above
print(df_clean['etf_type'].value_counts().to_string())

## 10. Build Strategy Subsets

We extract three focused sub-universes from the full deduplicated universe:

| DataFrame | Contents | Used for |
|-----------|----------|----------|
| `df_bonds` | All nominal bond ETFs (Treasury + Corporate + HY + Aggregate) | Bond strategy universe |
| `df_treasury` | US nominal Treasury ETFs only | Duration / yield-curve strategies |
| `df_sector` | Sector equity ETFs (one per GICS sector, most liquid) | Sector rotation strategy |

We print the full Treasury list and top 20 Sector ETFs sorted by AUM so you can **visually sanity-check** that the classification worked correctly before running any strategy on top of this data.

In [ ]:
# Nominal bond ETFs only — excludes TIPS (inflation-linked) and Munis (tax-exempt).
# Nominal bonds are driven purely by interest rate expectations, which makes them
# the cleanest instrument for yield-curve and duration strategies.
NOMINAL_BOND_TYPES = [
    'Bond - Treasury',    # US government nominal bonds
    'Bond - Corporate',   # investment-grade corporate bonds
    'Bond - High Yield',  # sub-investment-grade ("junk") bonds
    'Bond - Aggregate',   # blended bond market (e.g. AGG)
    'Bond - Other'        # catch-all nominal bonds not classified above
]

df_bonds    = df_clean[df_clean['etf_type'].isin(NOMINAL_BOND_TYPES)].copy()
df_treasury = df_clean[df_clean['etf_type'] == 'Bond - Treasury'].copy()
df_sector   = df_clean[df_clean['etf_type'] == 'Sector Equity'].copy()

print(f"Nominal Bond ETFs : {len(df_bonds)}")
print(f"  Treasury subset : {len(df_treasury)}")
print(f"Sector ETFs       : {len(df_sector)}")

# Print full lists — visually verify these look right before building strategy signals
print(f"\nAll Treasury ETFs (sorted by AUM):")
print(df_treasury[['ticker','fund_name','aum','net_expenses','ann_return','ann_vol']]
      .sort_values('aum', ascending=False).to_string(index=False))

print(f"\nTop 20 Sector ETFs by AUM:")
print(df_sector[['ticker','fund_name','aum','net_expenses','ann_return','ann_vol']]
      .sort_values('aum', ascending=False).head(20).to_string(index=False))

## 11. Build Wide Return Panels

We reshape the daily return data from **long format** (one row per ETF per day) into **wide format** (one column per ETF, one row per date):

```
Long format                         Wide format
─────────────────────────           ───────────────────────────────
permno | date       | ret_net       date       | SPY   | QQQ   | TLT
88292  | 2011-01-03 | 0.0012    →   2011-01-03 | 0.001 | 0.003 | -0.001
88292  | 2011-01-04 | -0.002        2011-01-04 | -0.002| 0.001 |  0.002
```

Wide format is the standard input for portfolio optimisation, factor models, and backtesting engines.

> **NaN values** in the panel are normal — they appear where an ETF didn't exist yet (launched after 2011) or had already been delisted. Strategy code must handle these appropriately (e.g. using `dropna()` or forward-filling only where valid).

In [ ]:
def return_panel(subset, daily_df, label=''):
    """
    Reshape long-format daily returns into a wide matrix (dates × tickers).

    Long format (input):           Wide format (output):
    ┌────────┬────────────┬───────┐  ┌────────────┬───────┬───────┬───────┐
    │ permno │ date       │ ret   │  │ date       │ SPY   │ QQQ   │ TLT   │
    ├────────┼────────────┼───────┤  ├────────────┼───────┼───────┼───────┤
    │ 84398  │ 2011-01-03 │ 0.001 │  │ 2011-01-03 │ 0.001 │ 0.003 │ -0.00 │
    │ 84398  │ 2011-01-04 │-0.002 │  │ 2011-01-04 │-0.002 │ 0.001 │  0.00 │
    └────────┴────────────┴───────┘  └────────────┴───────┴───────┴───────┘

    NaN appears where an ETF didn't exist yet or had been delisted — normal.
    """
    permnos = subset['permno'].dropna().astype(int).tolist()

    # pivot_table reshapes long → wide; aggregate function is 'mean' by default
    # but since each (permno, date) pair has exactly one return value, it just
    # passes the value through unchanged
    panel = (
        daily_df[daily_df['permno'].isin(permnos)]
        .pivot_table(index='date', columns='permno', values='ret_net')
    )

    # Replace numeric permno column headers with human-readable ticker symbols
    ticker_map    = subset.set_index('permno')['ticker'].to_dict()
    panel.columns = [ticker_map.get(int(c), str(c)) for c in panel.columns]

    nan_pct = panel.isna().mean().mean()
    print(f"{label}: {panel.shape[0]} days × {panel.shape[1]} ETFs  |  NaN%: {nan_pct:.1%}")
    return panel

treasury_rets = return_panel(df_treasury, daily_f, 'Treasury')
sector_rets   = return_panel(df_sector,   daily_f, 'Sector')
all_rets      = return_panel(df_clean,    daily_f, 'Full universe')

## 12. Save All Outputs

All cleaned dataframes and return panels are saved to `data/processed/` as **Parquet** files.

**Why Parquet instead of CSV?**
- Preserves exact data types (dates stay as dates, floats stay as floats — no silent conversions)
- ~10× faster to read than an equivalent CSV for large files
- Automatically compressed — smaller file size on disk

**Files saved:**

| File | Contents |
|------|----------|
| `etf_universe_clean.parquet` | Full deduplicated ETF universe with all metadata + computed stats |
| `etf_bonds.parquet` | Nominal bond ETF subset (Treasury + Corporate + HY + Aggregate) |
| `etf_treasury.parquet` | Treasury-only ETF subset |
| `etf_sector.parquet` | Sector equity ETF subset |
| `returns_treasury.parquet` | Wide return panel — Treasury ETFs (dates × tickers) |
| `returns_sector.parquet` | Wide return panel — Sector ETFs (dates × tickers) |
| `returns_all.parquet` | Wide return panel — full universe (dates × tickers) |

In [ ]:
# Save everything to the processed/ folder as Parquet files.
# Parquet is preferred over CSV because:
#   - It preserves exact data types (dates, floats, integers — no silent conversions)
#   - It is ~10× faster to read for large files
#   - It compresses data automatically, saving disk space
#
# index=False on metadata tables (DataFrames): the integer row index carries no
# information, so we don't save it.
# Return panels (matrices) use the default index=True since the date index IS data.

df_clean.to_parquet(     f"{PROC_DIR}/etf_universe_clean.parquet",   index=False)
df_bonds.to_parquet(     f"{PROC_DIR}/etf_bonds.parquet",            index=False)
df_treasury.to_parquet(  f"{PROC_DIR}/etf_treasury.parquet",         index=False)
df_sector.to_parquet(    f"{PROC_DIR}/etf_sector.parquet",           index=False)
treasury_rets.to_parquet(f"{PROC_DIR}/returns_treasury.parquet")
sector_rets.to_parquet(  f"{PROC_DIR}/returns_sector.parquet")
all_rets.to_parquet(     f"{PROC_DIR}/returns_all.parquet")

# Print a summary of what was saved and how large each file is
print("Saved to:", PROC_DIR, "\n")
for fname in ['etf_universe_clean','etf_bonds','etf_treasury','etf_sector',
              'returns_treasury','returns_sector','returns_all']:
    path    = f"{PROC_DIR}/{fname}.parquet"
    size_mb = os.path.getsize(path) / 1e6
    print(f"  {fname}.parquet  —  {size_mb:.1f} MB")